# Binder Parameter Analysis for Critical Temperature Estimation

## Overview
This notebook demonstrates the use of the Binder parameter ($U_4$) for estimating the critical temperature ($T_c$) of a diluted magnetic system with RKKY interactions using finite-size scaling.

The Binder parameter is a dimensionless universal quantity that provides a model-independent way to identify phase transitions regardless of system size.

## Section 1: Import Libraries and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j0, j1, y0, y1
from scipy.interpolate import interp1d
import json
from pathlib import Path
from tqdm import tqdm
import warnings

# Import local modules
from lattice import Lattice
from monte_carlo import MonteCarlo
from accumulator import Accumulator

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
warnings.filterwarnings('ignore', category=UserWarning)

print("✓ Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {j0.__module__.split('.')[0]}")

## Section 2: Plot RKKY Potential $V_{\text{RKKY}}(r \cdot k_F)$ for Multiple $k_F$ Values

The RKKY interaction in 2D is given by:
$$V_{\text{RKKY}}(r) = -J_0 [J_0(k_F r) Y_0(k_F r) + J_1(k_F r) Y_1(k_F r)]$$

where $J_n$ and $Y_n$ are Bessel functions of the first and second kind. The oscillations are a key feature of the RKKY mechanism for magnetic interactions.

In [ ]:
def rkky_potential(x, J0=1.0):
    """
    RKKY potential as function of dimensionless parameter x = k_F * r.
    
    V(x) = -J0 * [J0(x)*Y0(x) + J1(x)*Y1(x)]
    """
    x = np.asarray(x)
    x = np.where(x == 0, 1e-10, x)  # Avoid singularity at x=0
    return -J0 * (j0(x) * y0(x) + j1(x) * y1(x))

# Plot 1: RKKY potential as universal curve (fixed k_F * r scaling)
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('RKKY Potential and Physical Regimes', fontsize=16, fontweight='bold')

# Panel 1: Universal curve
x_range = np.linspace(0.1, 30, 1000)
v_rkky = rkky_potential(x_range)

ax = axes[0, 0]
ax.plot(x_range, v_rkky, 'b-', linewidth=2.5, label='$V_{RKKY}(x)$')
ax.axhline(0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.fill_between(x_range, 0, v_rkky, where=(v_rkky >= 0), alpha=0.2, color='red', label='Ferromagnetic')
ax.fill_between(x_range, 0, v_rkky, where=(v_rkky < 0), alpha=0.2, color='blue', label='Antiferromagnetic')
ax.set_xlabel('$x = k_F r$ (dimensionless)', fontsize=12, fontweight='bold')
ax.set_ylabel('$V_{RKKY}(x)$ / $J_0$', fontsize=12, fontweight='bold')
ax.set_title('Universal RKKY Oscillation', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])

# Panel 2: Real-space RKKY for different k_F
ax = axes[0, 1]
kf_values = [0.5, 1.0, 1.5, 2.0, 2.5]
r_max = 40
r = np.linspace(0.1, r_max, 1000)

for kf in kf_values:
    v = rkky_potential(kf * r)
    ax.plot(r, v, linewidth=2.5, label=f'$k_F = {kf}$ Å$^{{-1}}$', alpha=0.8)

ax.axhline(0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Distance $r$ (lattice units)', fontsize=12, fontweight='bold')
ax.set_ylabel('$V_{RKKY}(r)$ / $J_0$', fontsize=12, fontweight='bold')
ax.set_title('Real-Space RKKY Potential', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 20])

# Panel 3: Oscillation wavelength vs k_F
ax = axes[1, 0]
wavelengths = []
for kf in kf_values:
    # RKKY oscillation wavelength: λ ≈ π / k_F
    lambda_rkky = np.pi / kf
    wavelengths.append(lambda_rkky)

bars = ax.bar(range(len(kf_values)), wavelengths, color='steelblue', alpha=0.7, 
               edgecolor='black', linewidth=2)
ax.set_xticks(range(len(kf_values)))
ax.set_xticklabels([f'{kf}' for kf in kf_values])
ax.set_xlabel('Fermi Wavevector $k_F$ (Å$^{-1}$)', fontsize=12, fontweight='bold')
ax.set_ylabel('RKKY Wavelength $\lambda$ (lattice units)', fontsize=12, fontweight='bold')
ax.set_title('Oscillation Wavelength: $\lambda \sim \pi / k_F$', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Panel 4: First extrema locations
ax = axes[1, 1]
first_max = []
first_min = []
for kf in kf_values:
    # Approximate first extrema
    r_max_approx = np.pi / (2 * kf)
    r_min_approx = np.pi / kf
    first_max.append(r_max_approx)
    first_min.append(r_min_approx)

x_pos = np.arange(len(kf_values))
width = 0.35
ax.bar(x_pos - width/2, first_max, width, label='First Maximum', 
       color='coral', alpha=0.8, edgecolor='black', linewidth=1.5)
ax.bar(x_pos + width/2, first_min, width, label='First Minimum', 
       color='lightblue', alpha=0.8, edgecolor='black', linewidth=1.5)

ax.set_xticks(x_pos)
ax.set_xticklabels([f'{kf}' for kf in kf_values])
ax.set_xlabel('Fermi Wavevector $k_F$ (Å$^{-1}$)', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance $r$ (lattice units)', fontsize=12, fontweight='bold')
ax.set_title('Characteristic Length Scales', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plots/rkky_potential_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ RKKY potential plotted successfully")

## Section 3: Understand RKKY Oscillations and Physical Regimes

### Key Physical Concepts

**RKKY Mechanism**: The conduction electrons mediate magnetic interactions between localized spins, creating oscillatory patterns with alternating ferromagnetic (FM) and antiferromagnetic (AFM) regions.

**Fermi Wavevector and Carrier Density Relation**:
In 2D: $n = \frac{k_F^2}{\pi}$, so $k_F = \sqrt{\pi n}$

**Characteristic Scales**:
- Oscillation wavelength: $\lambda_F \sim \frac{\pi}{k_F}$
- Screening length: $\lambda_s \sim \frac{1}{k_F}$
- Interaction range: Multiple oscillations up to $\sim r \approx 10-20$ lattice units

**Physical Regimes**:
- **High $k_F$ (high doping)**: Short-wavelength oscillations, nearest-neighbor interactions dominated
- **Low $k_F$ (low doping)**: Long-wavelength oscillations, long-range interactions matter
- **Metal-Insulator Transition**: Around $n \approx 0.1-0.2$ (localization effects)

## Section 4: Initialize System Parameters from Physical Principles

In [ ]:
# Physical Parameters
# ==================

# Choose a reasonable doping concentration
doping = 0.25  # 25% occupancy - reasonable for diluted magnetic materials

# Compute Fermi wavevector from carrier density
# For 2D: n = k_F^2 / π
# For doping × lattice_size^2 electrons spread over area
carrie_density = doping  # Normalized carrier density
kf = np.sqrt(np.pi * carrie_density)

print(f"Physical Parameters:")
print(f"  Doping: {doping:.1%}")
print(f"  Carrier density: n = {carrie_density:.3f}")
print(f"  Fermi wavevector: k_F = {kf:.3f} Å⁻¹")
print(f"  RKKY wavelength: λ_F = π/k_F = {np.pi/kf:.2f} lattice units")
print()

# System sizes for finite-size scaling
# Should span multiple RKKY wavelengths
system_sizes = [(16, 16), (24, 24), (32, 32), (40, 40)]
print(f"System sizes for finite-size scaling: {system_sizes}")
print()

# Temperature range bracketing expected T_c
# For diluted RKKY systems: T_c ~ 0.1-0.5 J0 typically
T_min = 0.01
T_max = 1.0
n_temps = 12
temperatures = np.linspace(T_min, T_max, n_temps)

print(f"Temperature range: {T_min:.2f} - {T_max:.2f}")
print(f"Number of temperature points: {n_temps}")
print(f"Temperatures: {temperatures}")
print()

# MC simulation parameters
mc_params = {
    'warmup_steps': 2_000,   # Equilibration
    'production_steps': 5_000, # Data collection
    'method': 'metropolis',
    'seed': 42
}

print(f"MC Parameters:")
for key, val in mc_params.items():
    print(f"  {key}: {val}")

# Create output directory
Path('plots').mkdir(exist_ok=True)

print("\n✓ System parameters initialized")

## Section 5: Run Monte Carlo Simulations at Multiple Temperatures

This section runs Metropolis MC simulations for different system sizes across a range of temperatures. For each simulation, we collect magnetization moments ($\langle M^2 \rangle$, $\langle M^4 \rangle$) needed for the Binder parameter calculation.

In [ ]:
# ⚠️  WARNING: This cell takes several minutes to run!
# Store results: {(L, T) : {binder, error, moments}}
results = {}

print("=" * 70)
print("RUNNING MONTE CARLO SIMULATIONS")
print("=" * 70)
print()

for size_idx, (rows, cols) in enumerate(system_sizes, 1):
    L = rows  # System size
    print(f"\n[{size_idx}/{len(system_sizes)}] System: {L}×{cols}")
    print("-" * 70)
    
    results[L] = {}
    
    for T_idx, T in enumerate(temperatures):
        # Create lattice
        lattice = Lattice(
            rows=rows, cols=cols, 
            doping=doping, 
            kf=kf, 
            J0=1.0,  # Set J0 = 1 (energy scale)
            seed=mc_params['seed'] + T_idx * 1000 + size_idx * 10000
        )
        
        # Create MC simulator
        mc = MonteCarlo(lattice, progress=False)
        
        # Run simulation
        try:
            sim_info = mc.run_loop(
                warmup_steps=mc_params['warmup_steps'],
                steps=mc_params['production_steps'],
                T=T,
                method=mc_params['method']
            )
            
            # Compute Binder parameter and moments
            crit_info = mc.acc.compute_critical_exponents_info()
            
            results[L][T] = {
                'binder': crit_info['binder'],
                'binder_error': crit_info['binder_error'],
                'mean_m': crit_info['mean_m'],
                'mean_abs_m': crit_info['mean_abs_m'],
                'mean_m2': crit_info['mean_m2'],
                'mean_m4': crit_info['mean_m4'],
                'tau_E': sim_info['tau_E'],
                'tau_M': sim_info['tau_M'],
                'acceptance': sim_info['accept'],
            }
            
            status = f"✓ T={T:.3f}: U4={crit_info['binder']:.4f}±{crit_info['binder_error']:.4f}"
            
        except Exception as e:
            results[L][T] = {'error': str(e)}
            status = f"✗ T={T:.3f}: {str(e)[:40]}"
        
        print(f"  {status}")

print("\n" + "=" * 70)
print(f"✓ Simulations completed. Results stored for {len(results)} system sizes")
print("=" * 70)

## Section 6: Compute Binder Parameter and Estimate Critical Temperature

The Binder parameter is defined as:
$$U_4 = 1 - \frac{\langle M^4 \rangle}{3 \langle M^2 \rangle^2}$$

**Physical Significance**:
- **Paramagnetic phase** ($T > T_c$): $U_4 \approx 2/3$ (spins uncorrelated)
- **Ferromagnetic phase** ($T < T_c$): $U_4 \to 1$ (spins aligned)
- **At criticality** ($T = T_c$): $U_4$ reaches a universal value independent of system size
- **Intersection point**: Curves for different $L$ intersect at $T_c$ (finite-size scaling)

In [ ]:
# Plot Binder parameter vs temperature
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Extract data for plotting
sizes_plot = sorted([k for k in results.keys() if isinstance(k, int)])

fig.suptitle('Binder Parameter Analysis for Critical Temperature Estimation', 
             fontsize=14, fontweight='bold')

# Plot 1: Binder parameter curves
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(sizes_plot)))

T_curves = {}
U4_curves = {}

for size_idx, L in enumerate(sizes_plot):
    T_list = []
    U4_list = []
    U4_err = []
    
    for T in sorted(results[L].keys()):
        if 'error' not in results[L][T]:
            T_list.append(T)
            U4_list.append(results[L][T]['binder'])
            U4_err.append(results[L][T]['binder_error'])
    
    T_curves[L] = np.array(T_list)
    U4_curves[L] = np.array(U4_list)
    
    # Plot with error bars
    ax.errorbar(T_list, U4_list, yerr=U4_err, marker='o', markersize=8,
                linewidth=2.5, label=f'L = {L}', color=colors[size_idx], 
                capsize=5, capthick=2, alpha=0.8)

# Add reference lines
ax.axhline(2/3, color='green', linestyle='--', linewidth=2, alpha=0.5, 
           label='Param. limit ($U_4 = 2/3$)')
ax.axhline(1.0, color='red', linestyle='--', linewidth=2, alpha=0.5, 
           label='Ordered limit ($U_4 = 1$)')

ax.set_xlabel('Temperature $T$ / $J_0$', fontsize=12, fontweight='bold')
ax.set_ylabel('Binder Parameter $U_4$', fontsize=12, fontweight='bold')
ax.set_title('Finite-Size Scaling of $U_4(T)$', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
ax.set_ylim([0.6, 1.05])

# Plot 2: Zoom in on crossing region
ax = axes[1]

for size_idx, L in enumerate(sizes_plot):
    T_list = []
    U4_list = []
    U4_err = []
    
    for T in sorted(results[L].keys()):
        if 'error' not in results[L][T]:
            T_list.append(T)
            U4_list.append(results[L][T]['binder'])
            U4_err.append(results[L][T]['binder_error'])
    
    ax.errorbar(T_list, U4_list, yerr=U4_err, marker='o', markersize=8,
                linewidth=2.5, label=f'L = {L}', color=colors[size_idx],
                capsize=5, capthick=2, alpha=0.8)

ax.set_xlabel('Temperature $T$ / $J_0$', fontsize=12, fontweight='bold')
ax.set_ylabel('Binder Parameter $U_4$', fontsize=12, fontweight='bold')
ax.set_title('Zoomed View: Curve Intersections', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/binder_parameter_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Binder parameter plots created")

In [ ]:
# Estimate T_c from curve intersections
print("=" * 70)
print("CRITICAL TEMPERATURE ESTIMATION")
print("=" * 70)
print()

# Method 1: Find intersection of two largest systems
if len(sizes_plot) >= 2:
    L1 = sizes_plot[-2]  # Second largest
    L2 = sizes_plot[-1]  # Largest
    
    # Interpolate curves
    T1 = T_curves[L1]
    U4_1 = U4_curves[L1]
    T2 = T_curves[L2]
    U4_2 = U4_curves[L2]
    
    # Find intersection by linear interpolation
    f1 = interp1d(T1, U4_1, kind='cubic', fill_value='extrapolate')
    f2 = interp1d(T2, U4_2, kind='cubic', fill_value='extrapolate')
    
    # Search for intersection in temperature range
    T_search = np.linspace(T1.min(), T1.max(), 200)
    diff = f1(T_search) - f2(T_search)
    
    # Find sign change
    sign_changes = np.where(np.diff(np.sign(diff)))[0]
    
    T_c_estimates = []
    for idx in sign_changes:
        T_c = (T_search[idx] + T_search[idx + 1]) / 2
        T_c_estimates.append(T_c)
    
    if T_c_estimates:
        # Use first intersection (closest to middle of range)
        T_c = T_c_estimates[0]
        print(f"Intersection of L={L1} and L={L2}:")
        print(f"  T_c ≈ {T_c:.4f} J₀")
        print(f"  (All intersection points: {', '.join([f'{t:.4f}' for t in T_c_estimates])})")
    else:
        T_c = None
        print("⚠ No intersection found in temperature range")
else:
    T_c = None
    print("⚠ Insufficient system sizes for intersection finding")

print()

# Method 2: Summary statistics at each temperature
print("Binder Parameter Summary:")
print("-" * 70)
print(f"{'T (J₀)':>10} | ", end='')
for L in sizes_plot:
    print(f"{'L=' + str(L):>15} | ", end='')
print()
print("-" * 70)

for T in sorted(temperatures):
    print(f"{T:>10.4f} | ", end='')
    for L in sizes_plot:
        if T in results[L] and 'binder' in results[L][T]:
            u4 = results[L][T]['binder']
            err = results[L][T]['binder_error']
            print(f"{u4:.4f}±{err:.4f} | ", end='')
        else:
            print(f"{'N/A':>15} | ", end='')
    print()

print("=" * 70)
if T_c is not None:
    print(f"\n✓ Estimated Critical Temperature: T_c ≈ {T_c:.4f} J₀")
    print(f"  This represents the scale-invariant point where magnetic")
    print(f"  correlations diverge in the thermodynamic limit.")
else:
    print("\n⚠ T_c estimation inconclusive - more data points recommended")
print()

## Section 7: Analyze Finite-Size Scaling and Universality

Finite-size scaling is a powerful technique in critical phenomena that allows us to extract critical exponents and locate the true critical point from simulations on finite systems.

In [ ]:
# Compute critical exponents and universality
print("=" * 70)
print("FINITE-SIZE SCALING ANALYSIS")
print("=" * 70)
print()

# Summary of system properties
print("Physical Properties:")
print(f"  Doping: {doping:.1%}")
print(f"  Fermi wavevector: k_F = {kf:.3f}")
print(f"  RKKY wavelength: λ_F = {np.pi/kf:.2f} lattice units")
print()

print("System Sizes and Characteristics:")
print("-" * 70)
print(f"{'L':>6} | {'L²':>6} | {'λ_F/L':>8} | {'Wavelengths':>12}")
print("-" * 70)

for L in sizes_plot:
    l_squared = L * L
    ratio = (np.pi / kf) / L
    wavelengths = L / (np.pi / kf)
    print(f"{L:>6} | {l_squared:>6} | {ratio:>8.3f} | {wavelengths:>12.2f}")

print()
print("Critical Temperature Information:")
print("-" * 70)

if T_c is not None:
    # Evaluate universal value at T_c
    print(f"Estimated T_c = {T_c:.4f} J₀")
    print()
    
    # Interpolate U_4 value at T_c for largest system
    L_largest = sizes_plot[-1]
    if L_largest in U4_curves:
        f_largest = interp1d(T_curves[L_largest], U4_curves[L_largest], 
                            kind='cubic', fill_value='extrapolate')
        U4_at_Tc = f_largest(T_c)
        print(f"Binder parameter at T_c:")
        print(f"  U_4(T_c) ≈ {U4_at_Tc:.4f}")
        print()
        
        # Note on universality
        print("Universality Class:")
        print(f"  {"✓ FM ordering detected" if U4_at_Tc > 0.85 else "✗ Critical behavior inconclusive"}")
        print(f"  Curve intersection provides scale-invariant T_c")
        print()

# Create comprehensive summary figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Finite-Size Scaling Analysis', fontsize=14, fontweight='bold')

# Panel 1: Binder parameter with intersection
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(sizes_plot)))

for size_idx, L in enumerate(sizes_plot):
    T_list = sorted([T for T in results[L].keys() if 'error' not in results[L][T]])
    U4_list = [results[L][T]['binder'] for T in T_list]
    
    ax.plot(T_list, U4_list, marker='o', linewidth=2.5, markersize=8,
            color=colors[size_idx], label=f'L = {L}', alpha=0.8)

if T_c is not None:
    ax.axvline(T_c, color='red', linestyle='--', linewidth=2.5, 
               label=f'$T_c$ ≈ {T_c:.4f}', alpha=0.7)

ax.axhline(2/3, color='green', linestyle=':', linewidth=1.5, alpha=0.5)
ax.axhline(1.0, color='blue', linestyle=':', linewidth=1.5, alpha=0.5)

ax.set_xlabel('$T / J_0$', fontsize=12, fontweight='bold')
ax.set_ylabel('$U_4$', fontsize=12, fontweight='bold')
ax.set_title('Binder Parameter Scaling', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 2: Magnetization vs T
ax = axes[1]

for size_idx, L in enumerate(sizes_plot):
    T_list = sorted([T for T in results[L].keys() if 'error' not in results[L][T]])
    M2_list = [results[L][T]['mean_m2'] for T in T_list]
    
    ax.plot(T_list, M2_list, marker='s', linewidth=2.5, markersize=8,
            color=colors[size_idx], label=f'L = {L}', alpha=0.8)

if T_c is not None:
    ax.axvline(T_c, color='red', linestyle='--', linewidth=2.5, alpha=0.7)

ax.set_xlabel('$T / J_0$', fontsize=12, fontweight='bold')
ax.set_ylabel('$\\langle M^2 \\rangle$', fontsize=12, fontweight='bold')
ax.set_title('Magnetization Moment', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 3: Summary table
ax = axes[2]
ax.axis('off')

summary_text = "Summary Statistics\n" + "=" * 40 + "\n\n"
summary_text += f"Doping: {doping:.1%}\n"
summary_text += f"$k_F$: {kf:.4f} Å⁻¹\n"
summary_text += f"$\lambda_F$: {np.pi/kf:.2f} lat. units\n\n"

if T_c is not None:
    summary_text += f"Estimated $T_c$: {T_c:.4f} J₀\n\n"

summary_text += f"System sizes: {', '.join([str(L) for L in sizes_plot])}\n"
summary_text += f"Temperatures: {len(temperatures)} points\n"
summary_text += f"MC steps: {mc_params['warmup_steps'] + mc_params['production_steps']}\n\n"

summary_text += "Universality:\n"
summary_text += "• Binder parameter is\n"
summary_text += "  dimensionless and universal\n"
summary_text += "• Intersection point gives $T_c$\n"
summary_text += "• Value at $T_c$ independent of $L$\n"

ax.text(0.1, 0.95, summary_text, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('plots/finite_size_scaling_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Finite-size scaling analysis complete")

## Summary and Interpretation

### Key Results

1. **RKKY Interactions**: The oscillatory RKKY potential exhibits characteristic length scales $\lambda_F \sim 1/k_F$, which scale with carrier doping.

2. **Binder Parameter Method**: By computing $U_4 = 1 - \langle M^4 \rangle / (3\langle M^2 \rangle^2)$ for multiple system sizes, we can identify the critical temperature through curve intersections without finite-size bias.

3. **Critical Temperature Estimation**: The intersection point of $U_4(T)$ curves for different system sizes provides an estimate of $T_c$ in the thermodynamic limit.

4. **Universality**: The Binder parameter is a dimensionless universal quantity that converges to a universal value at criticality, independent of microscopic details.

### Physical Interpretation

- **Paramagnetic phase** ($T > T_c$): Spins are uncorrelated, $U_4 \approx 2/3$
- **Ferromagnetic phase** ($T < T_c$): Spins are aligned, $U_4 \to 1$
- **Critical point** ($T = T_c$): Universal scaling behavior emerges

### Advantages of This Method

✓ Model-independent approach to identifying phase transitions  
✓ Works for any universality class  
✓ Provides finite-size scaling information  
✓ No need to know critical exponents a priori  
✓ Robust to finite-size effects through curve intersection  

### References

1. Binder, K. (1981). "Critical properties from Monte Carlo coarse graining and renormalization." Phys. Rev. B **23**, 3344.
2. RKKY theory and diluted magnetic systems in condensed matter physics
3. Finite-size scaling in critical phenomena